In [1]:
pip install pandas pyarrow


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time
import json
import pandas as pd

def mb(path):
    return os.path.getsize(path) / (1024 * 1024)

def log(msg):
    print(msg)

def main():
    os.makedirs("data_lake", exist_ok=True)

    N = 200_000

    csv_path = "data_lake/clientes.csv"
    jsonl_path = "data_lake/clientes.jsonl"
    parquet_path = "data_lake/clientes.parquet"

    log("=== AULA 5 — DATA LAKE E FORMATOS ===")
    log(f"Gerando dataset com {N:,} linhas...\n")

    t0 = time.perf_counter()
    df = pd.DataFrame({
        "id": range(1, N + 1),
        "nome": [f"Cliente_{i}" for i in range(1, N + 1)],
        "cidade": ["São Paulo", "Rio de Janeiro", "Curitiba", "BH", "Salvador"] * (N // 5),
        "valor": [float(i % 5000) for i in range(N)]
    })
    t1 = time.perf_counter()
    log(f"[GERAÇÃO] OK | tempo: {t1 - t0:.2f}s | linhas: {len(df):,}\n")

    t0 = time.perf_counter()
    df.to_csv(csv_path, index=False)
    t1 = time.perf_counter()
    log(f"[WRITE CSV] tempo: {t1 - t0:.2f}s | tamanho: {mb(csv_path):.1f} MB")

    t0 = time.perf_counter()
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for row in df.to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    t1 = time.perf_counter()
    log(f"[WRITE JSONL] tempo: {t1 - t0:.2f}s | tamanho: {mb(jsonl_path):.1f} MB")

    t0 = time.perf_counter()
    df.to_parquet(parquet_path, index=False)
    t1 = time.perf_counter()
    log(f"[WRITE PARQUET] tempo: {t1 - t0:.2f}s | tamanho: {mb(parquet_path):.1f} MB\n")

    t0 = time.perf_counter()
    _ = pd.read_csv(csv_path)
    t1 = time.perf_counter()
    log(f"[READ CSV] tempo: {t1 - t0:.2f}s")

    t0 = time.perf_counter()
    _ = pd.read_json(jsonl_path, lines=True)
    t1 = time.perf_counter()
    log(f"[READ JSONL] tempo: {t1 - t0:.2f}s")

    t0 = time.perf_counter()
    _ = pd.read_parquet(parquet_path)
    t1 = time.perf_counter()
    log(f"[READ PARQUET] tempo: {t1 - t0:.2f}s")

    log("\n=== FIM ===")

if __name__ == "__main__":
    main()

=== AULA 5 — DATA LAKE E FORMATOS ===
Gerando dataset com 200,000 linhas...

[GERAÇÃO] OK | tempo: 0.09s | linhas: 200,000

[WRITE CSV] tempo: 0.24s | tamanho: 7.3 MB
[WRITE JSONL] tempo: 1.19s | tamanho: 15.3 MB
[WRITE PARQUET] tempo: 0.29s | tamanho: 2.2 MB

[READ CSV] tempo: 0.15s
[READ JSONL] tempo: 0.57s
[READ PARQUET] tempo: 0.32s

=== FIM ===
